In [ ]:
import pandas as pd
import nltk
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import spacy
from spacy import displacy
from IPython.display import HTML
from nltk.tokenize import word_tokenize, sent_tokenize

LOAD DATA 

In [8]:
bbc_news = pd.read_csv('bbc_news.csv')
bbc_news.head()

,Unnamed: 0,index,title,pubDate,guid,link,description
0,0,6684,Can I refuse to work?,"Wed, 10 Aug 2022 15:46:18 GMT",https://www.bbc.co.uk/news/business-62147992,https://www.bbc.co.uk/news/business-62147992?a...,With much of the UK enduring another period of...
1,1,9267,'Liz Truss the Brief?' World reacts to UK poli...,"Mon, 17 Oct 2022 11:35:12 GMT",https://www.bbc.co.uk/news/world-63285480,https://www.bbc.co.uk/news/world-63285480?at_m...,The UK's political chaos has been watched arou...
2,2,7387,Rationing energy is nothing new for off-grid c...,"Wed, 31 Aug 2022 05:20:18 GMT",https://www.bbc.co.uk/news/uk-scotland-highlan...,https://www.bbc.co.uk/news/uk-scotland-highlan...,Scoraig in the north west Highlands has long h...
3,3,767,The hunt for superyachts of sanctioned Russian...,"Tue, 22 Mar 2022 14:37:01 GMT",https://www.bbc.co.uk/news/60739336,https://www.bbc.co.uk/news/60739336?at_medium=...,"Wealthy Russians sanctioned by the US, EU and ..."
4,4,3712,Platinum Jubilee: 70 years of the Queen in 70 ...,"Wed, 01 Jun 2022 23:17:33 GMT",https://www.bbc.co.uk/news/uk-61660128,https://www.bbc.co.uk/news/uk-61660128?at_medi...,A quick look back at the Queen's 70 years on t...


In [21]:
bbc_titles = pd.DataFrame(bbc_news['title'])
bbc_titles.head()

,title
0,Can I refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...
2,Rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...


Text Cleaning

In [43]:
#Convert text to lower case
bbc_titles['lowercased_titles'] = bbc_titles['title'].str.lower()

#remove stopwords
en_stopwords = stopwords.words('english')
bbc_titles['title_nostopwords'] = bbc_titles['lowercased_titles'].apply(lambda x: " ".join([word for word in x.split() if word not in en_stopwords]))

#remove punctuations
bbc_titles["titles_no_punct"] = bbc_titles['title_nostopwords'].apply(lambda x: re.sub(r"[^\w\s]", '', x))

#tokenize
bbc_titles['tokenized_clean_titles'] = bbc_titles['titles_no_punct'].apply(lambda x: word_tokenize(x))
bbc_titles['tokenized_raw_titles'] = bbc_titles['title'].apply(lambda x: word_tokenize(x))

#lematize
lematizer = WordNetLemmatizer()
bbc_titles['lematized_titles'] = bbc_titles['tokenized_titles'].apply(lambda x: [lematizer.lemmatize(word) for word in x ])

bbc_titles.head()

,title,lowercased_titles,title_nostopwords,titles_no_punct,tokenized_titles,lematized_titles,tokenized_clean_titles,tokenized_raw_titles
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[refuse, work]","[refuse, work]","[refuse, work]","[Can, I, refuse, to, work, ?]"
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil,"[liz, truss, brief, world, reacts, uk, politic...","[liz, truss, brief, world, reacts, uk, politic...","[liz, truss, brief, world, reacts, uk, politic...","['Liz, Truss, the, Brief, ?, ', World, reacts,..."
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community,"[rationing, energy, nothing, new, offgrid, com...","[rationing, energy, nothing, new, offgrid, com...","[rationing, energy, nothing, new, offgrid, com...","[Rationing, energy, is, nothing, new, for, off..."
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs,"[hunt, superyachts, sanctioned, russian, oliga...","[hunt, superyachts, sanctioned, russian, oliga...","[hunt, superyachts, sanctioned, russian, oliga...","[The, hunt, for, superyachts, of, sanctioned, ..."
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds,"[platinum, jubilee, 70, years, queen, 70, seco...","[platinum, jubilee, 70, year, queen, 70, second]","[platinum, jubilee, 70, years, queen, 70, seco...","[Platinum, Jubilee, :, 70, years, of, the, Que..."


In [45]:
tokens_clean_list = sum(bbc_titles['tokenized_clean_titles'], [])
tokens_raw_list = sum(bbc_titles['tokenized_raw_titles'], [])

POS Tagging

In [ ]:
nlp = spacy.load('en_core_web_sm')

tokens_raw = " ".join(tokens_raw_list)

spacy_doc = nlp(tokens_raw)

pos_list = [{'Tokens':word.text, 'Parts Of Speech':word.pos_} for word in spacy_doc]

parts_of_speech_df = pd.DataFrame(pos_list)

,Tokens,Parts Of Speech
0,Can,AUX
1,I,PRON
2,refuse,VERB
3,to,PART
4,work,VERB


In [103]:
#token frequency 
token_frequecy = parts_of_speech_df.groupby(['Tokens', 'Parts Of Speech']).size().reset_index(name='Counts').sort_values(by='Counts', ascending=False)
token_frequecy.head(15)

# most common nouns
nouns = token_frequecy[token_frequecy['Parts Of Speech'] == 'NOUN'][:10]
nouns

# most common verbs
verbs = token_frequecy[token_frequecy['Parts Of Speech'] == 'VERB'][:10]
verbs

# most common adj
adj = token_frequecy[token_frequecy['Parts Of Speech'] == 'ADJ'][:10]
adj

,Tokens,Parts Of Speech,Counts
3244,new,ADJ,28
1400,Russian,ADJ,21
2606,final,ADJ,16
19,-,ADJ,14
2625,first,ADJ,12
3199,more,ADJ,10
1994,big,ADJ,9
2835,high,ADJ,9
3000,last,ADJ,8
3304,other,ADJ,8


NER

In [130]:
tokens_clean = " ".join(tokens_clean_list)
spacy_ner = nlp(tokens_clean)

er_list = [{'Entity':entity.text, 'Entity Title': entity.label_} for entity in spacy_ner.ents if pd.isna(entity.label_) is False]
er_df = pd.DataFrame(er_list)

In [128]:
html = displacy.render(spacy_ner, style='ent', jupyter=False)
display(HTML(html))